# Hands-on: Cumulant kernel with Sympy

<div style="
    background-color: #f8f9fa; 
    border-left: 5px solid #ced4da; 
    padding: 15px; 
    margin: 10px 0; 
    border-radius: 4px; 
    color: #495057;
">
    <strong>Reference:</strong> Notebook can be found at <a href="https://github.com/hayashi-workshop/clbmtaichi/blob/main/generator/handson_derivation_cumulant.ipynb">generator/handson_derivation_cumulant.ipynb</a>
</div>

We shall first review the algorithm to understand what we need in construction of cumulant LBM.

- Forward transformation from $f$ to $m$
- Forward transformation from $m$ to $\kappa$
- Forward transformation from $\kappa$ to $C$
- Collision in cumulant space ($C \rightarrow C^{*}$)
- Backward transformation from $C^{*}$ to $\kappa^{*}$
- Backward transformation from $\kappa^{*}$ to $m^{*}$
- Backward transformation from $m^{*}$ to $f^{*}$

We need the forward transformation pipeline ($f \rightarrow m \rightarrow \kappa \rightarrow C$) and the backward transformation pipeline ($C \rightarrow \kappa \rightarrow m \rightarrow f$); seems many, but actually the fact that cumulants of 2nd and 3rd orders are the same with $\kappa$ reduces implementation burden. 

Here, we try to develop D3Q27 version; however, we also implement D2Q9 version in utility functions. 

<div style="
    background-color: #f0f8ff; 
    border-left: 5px solid #a2c4c9; 
    padding: 15px; 
    margin: 10px 0; 
    border-radius: 4px; 
    color: #2c4a5e;
">
    <strong>Note:</strong> Fast transformation (Geier et al., 2015) itself is already optimized. Sympy CSE seems not so effective for the transformation algorithms. Reduced operation vs. Register pressure; this point should be further studied.  
</div>

Import required packages in this hands-on, which does not depend on external libraries except for `sympy` and `numpy`. 

In [1]:
import sympy as sp
from sympy import symbols, Eq

import itertools
import math
import numpy as np

## Weights

Get started from the weights as we did for the BGK case. `calculate_weight` is the same as in BGK.

In [2]:
def calculate_weight(vec):
    w_base = {0: sp.Rational(2, 3), 1: sp.Rational(1, 6), -1: sp.Rational(1, 6)}
    weight = 1
    for component in vec:
        weight *= w_base[component]
    return weight

## Velocity sets

`create_vectors` we developed previously can also be used; however, we extend it to include D3Q27 option. D2Q9 consists of all possible combinations of $-1$, $0$, $+1$, so that `itertools.product([-1, 0, 1], repeat=dim)` simply gave us all velocity sets. Fortunately, D3Q27 has the same feature, and `itertools` provides all velocities for `dim = 3` as well. Only problem we need to consider is the *order* of the velocities. In D3Q27, the order of the lattice velocities is defined as *1st velocity, oppsite to 1st velocity is put on the second place, 3rd velocity, oppsite to 3rd velocity is put on the 4th place...* This is totally different from the conventional D2Q9 we implemented before. Let's see the ordering below: 

In [3]:
def create_vectors(dim):
    all_vectors = list(itertools.product([-1, 0, 1], repeat=dim))

    vectors = []

    if dim == 2: # * this part is the same with the BGK hands-on * #
        def lbm_2d_sort_key(vec):
            length_sq = sum(c**2 for c in vec) # cx^2 + cy^2 
            
            if length_sq == 0: return (0, -100) # c0
                
            angle = math.atan2(vec[1], vec[0]) # angle between c and x-axis
            if angle < 0: angle += 2 * math.pi # quadrant correction
    
            return (length_sq, angle)
    
        # sort for (length_sq, angle). (0,0) is placed first since its length is 0. Next, four components of length_sq = 1 are sorted for angle. Then, ..
        sorted_pairs = sorted([(v, calculate_weight(v)) for v in all_vectors], key=lambda x: lbm_2d_sort_key(x[0]))
    
        # pick sorted vector from the result and list them. 
        vectors = [p[0] for p in sorted_pairs]
        
    elif dim == 3: 
#       2- 0 -1 # <- anti-direction pairing ->
        
        groups = {}

        for v in all_vectors:
            w = calculate_weight(v)
            groups.setdefault(w, []).append(v) # velocities grouped by the value of weight

        vectors.append((0, 0, 0)) # put center velocity first
        
        for w in sorted(groups.keys(), reverse=True): # search velocities grouped by the value of w; from 8/27, 2/27, 1/54, to 1/216

            if w == sp.Rational(8, 27): continue # skip center; it is already on the top
                
            unvisited = list(groups[w]) # fetch velocities for current w 
            unvisited.sort(key=lambda x: (tuple(abs(c) for c in x), x), reverse=True) # sort list so as to have positive one first; (1,0,0) is expected to be found earlier than (-1,0,0)
            
            while unvisited: # search pair and register them
                v1 = unvisited.pop(0) # fetch next velocity from pool
                v2 = tuple(-c for c in v1) # set pair 
                unvisited.remove(v2) # remove from pool since it has been found
                vectors.append(v1) 
                vectors.append(v2)
    
    return vectors

In [4]:
dim = 3

vectors = create_vectors(dim)

weights = [calculate_weight(v) for v in vectors]

Let's check the result. In order to minimize the display space, we put the vectors in matrix form.

In [5]:
sp.Matrix(vectors).T # the most left is (0,0,0) for c0, the next one is (1,0,0) for c1,...

Matrix([
[0, 1, -1, 0,  0, 0,  0, 1, -1,  1, -1, 1, -1,  1, -1, 0,  0,  0,  0, 1, -1,  1, -1,  1, -1,  1, -1],
[0, 0,  0, 1, -1, 0,  0, 1, -1, -1,  1, 0,  0,  0,  0, 1, -1,  1, -1, 1, -1,  1, -1, -1,  1, -1,  1],
[0, 0,  0, 0,  0, 1, -1, 0,  0,  0,  0, 1, -1, -1,  1, 1, -1, -1,  1, 1, -1, -1,  1,  1, -1, -1,  1]])

In [6]:
sum(weights) # sum w = 1: the scalar constrant for the weight vector

1

Very good. For the use in the following calulations, 

In [7]:
num_pops = len(vectors) # number of populations = 9 or 27

## Macroscopic variables

Define distribution functions and macroscopic variables. 

In [8]:
f_syms = sp.symbols([f'f{i}' for i in range(num_pops)]) # sympy symbol of f

In [9]:
rho_name = 'rho'
rho = sp.Symbol(rho_name)

vel_names = ['u', 'v', 'w'][:dim] # slicing eliminate 'w' when dim=2. 
vel_syms  = sp.symbols(vel_names)

rho_expr = sum(f_syms)

vel_exprs = []
for d_idx in range(dim): 
    momentum_expr = sum(vec[d_idx] * f_syms[i] for i, vec in enumerate(vectors))
    vel_exprs.append( momentum_expr / ( rho_expr ) ) 

display(Eq(rho, rho_expr))
display(Eq(rho * vel_syms[1], rho_expr * vel_exprs[1]))

Eq(rho, f0 + f1 + f10 + f11 + f12 + f13 + f14 + f15 + f16 + f17 + f18 + f19 + f2 + f20 + f21 + f22 + f23 + f24 + f25 + f26 + f3 + f4 + f5 + f6 + f7 + f8 + f9)

Eq(rho*v, f10 + f15 - f16 + f17 - f18 + f19 - f20 + f21 - f22 - f23 + f24 - f25 + f26 + f3 - f4 + f7 - f8 - f9)

## Forward transformation: $f \rightarrow m$

The forward transformation from $f$ to $m$ is defined by $m = M f$. This is the general definition, we have not yet defined the concrete form of the transformation matrix $M$. We follow the implementation of Geier et al (2015), that is, $m$ is the raw moment given by $m_{\alpha \beta \gamma} = \sum_{i=0}^{Q-1} c_{ix}^{\alpha} c_{iy}^{\beta} c_{iz}^{\gamma} f_{i}$. Some examples are shown below: 

- For $(0,0,0)$, $m_{000} = \sum_{i=0}^{Q-1} f_{i}$. Hence, all componets in the corresponding row of $M$ is $1$.
- For $(1,0,0)$, $m_{100} = \sum_{i=0}^{Q-1} c_{ix} f_{i}$.
- For $(0,2,0)$, $m_{020} = \sum_{i=0}^{Q-1} c_{iy}^{2} f_{i}$.

It is convenient to have a list of all combinations of $\alpha \beta \gamma$ to set the order of the moments. 

In [10]:
all_orders = list(itertools.product((0, 1, 2), repeat=dim))

moment_orders = sorted(all_orders, key=lambda x: (sum(x), -x[0]))

In [11]:
sp.Matrix(all_orders).T

Matrix([
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2],
[0, 0, 0, 1, 1, 1, 2, 2, 2, 0, 0, 0, 1, 1, 1, 2, 2, 2, 0, 0, 0, 1, 1, 1, 2, 2, 2],
[0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2]])

In [12]:
sp.Matrix(moment_orders).T

Matrix([
[0, 1, 0, 0, 2, 1, 1, 0, 0, 0, 2, 2, 1, 1, 1, 0, 0, 2, 2, 2, 1, 1, 0, 2, 2, 1, 2],
[0, 0, 0, 1, 0, 0, 1, 0, 1, 2, 0, 1, 0, 1, 2, 1, 2, 0, 1, 2, 1, 2, 2, 1, 2, 2, 2],
[0, 0, 1, 0, 0, 1, 0, 2, 1, 0, 1, 0, 2, 1, 0, 2, 1, 2, 1, 0, 2, 1, 2, 2, 1, 2, 2]])

Set moment names, which are used later.

In [13]:
mom_names = ["m" + "".join(map(str, order)) for order in moment_orders]

### Standard transformation ($f \rightarrow m$)

<div style="
    background-color: #f0f8ff; 
    border-left: 5px solid #a2c4c9; 
    padding: 15px; 
    margin: 10px 0; 
    border-radius: 4px; 
    color: #2c4a5e;
">
    <strong>Note:</strong> This will not be used in the actual implementation, so you can skip this section.
</div>

#### Matrix construction

It should be noted that the order of moments is *not* related to the order of $\mathbf{c}$. We create $m_{\alpha \beta \gamma}$ for this order `moment_orders`. `create_trans_matrix` gives you $M$, the raw order of which is compatible with `moment_orders`. 

In [14]:
def create_trans_matrix(moment_orders, vectors):
    M_list = []
    for order in moment_orders: # order is tuple of (\alpha, \beta, \gamma) running from (0,0,0) to (2,2,2)
        row = [] # compute components row by row
        for vec in vectors: # vector c
            term = 1
            for c, alpha in zip(vec, order): 
                term *= (c ** alpha) # c_{ix}^{alpha} c_{iy}^{beta} c_{iz}^{gamma}
            row.append(term)
        M_list.append(row)
    M = sp.Matrix(M_list) # converted as sympy Matrix
    return M, M.inv() # return M and its inverse

In [15]:
M, M_inv = create_trans_matrix(moment_orders, vectors)

$M$ is *sparse*, meaning that most components are $0$!

In [16]:
M

Matrix([
[1, 1,  1, 1,  1, 1,  1, 1,  1,  1,  1, 1,  1,  1,  1, 1,  1,  1,  1, 1,  1,  1,  1,  1,  1,  1,  1],
[0, 1, -1, 0,  0, 0,  0, 1, -1,  1, -1, 1, -1,  1, -1, 0,  0,  0,  0, 1, -1,  1, -1,  1, -1,  1, -1],
[0, 0,  0, 0,  0, 1, -1, 0,  0,  0,  0, 1, -1, -1,  1, 1, -1, -1,  1, 1, -1, -1,  1,  1, -1, -1,  1],
[0, 0,  0, 1, -1, 0,  0, 1, -1, -1,  1, 0,  0,  0,  0, 1, -1,  1, -1, 1, -1,  1, -1, -1,  1, -1,  1],
[0, 1,  1, 0,  0, 0,  0, 1,  1,  1,  1, 1,  1,  1,  1, 0,  0,  0,  0, 1,  1,  1,  1,  1,  1,  1,  1],
[0, 0,  0, 0,  0, 0,  0, 0,  0,  0,  0, 1,  1, -1, -1, 0,  0,  0,  0, 1,  1, -1, -1,  1,  1, -1, -1],
[0, 0,  0, 0,  0, 0,  0, 1,  1, -1, -1, 0,  0,  0,  0, 0,  0,  0,  0, 1,  1,  1,  1, -1, -1, -1, -1],
[0, 0,  0, 0,  0, 1,  1, 0,  0,  0,  0, 1,  1,  1,  1, 1,  1,  1,  1, 1,  1,  1,  1,  1,  1,  1,  1],
[0, 0,  0, 0,  0, 0,  0, 0,  0,  0,  0, 0,  0,  0,  0, 1,  1, -1, -1, 1,  1, -1, -1, -1, -1,  1,  1],
[0, 0,  0, 1,  1, 0,  0, 1,  1,  1,  1, 0,  0,  0,  0, 1,  1,  1,  1, 1, 

#### Get into moment space

Simply, $m = M f$:

In [17]:
m_exprs = M@sp.Matrix(f_syms)

Display the moments in the 0th and 2nd places: 

In [18]:
display( Eq(sp.Symbol(mom_names[0]), m_exprs[0]) )
display( Eq(sp.Symbol(mom_names[2]), m_exprs[2]) )

Eq(m000, f0 + f1 + f10 + f11 + f12 + f13 + f14 + f15 + f16 + f17 + f18 + f19 + f2 + f20 + f21 + f22 + f23 + f24 + f25 + f26 + f3 + f4 + f5 + f6 + f7 + f8 + f9)

Eq(m001, f11 - f12 - f13 + f14 + f15 - f16 - f17 + f18 + f19 - f20 - f21 + f22 + f23 - f24 - f25 + f26 + f5 - f6)

We know that these are $\rho$ and $\rho w$, so validate the results by comparing them with the macro variables we have already computed. 

In [19]:
m_raw = {}

for i in range(len(moment_orders)):
    o, name = moment_orders[i], mom_names[i]
    m_raw[o] = m_exprs[i]

Eq(sp.Symbol('m' + '2'*dim), m_raw[(2,2,2)[:dim]])

Eq(m222, f19 + f20 + f21 + f22 + f23 + f24 + f25 + f26)

### Chimera fast transformation ($f_{i} \rightarrow m_{\alpha \beta \gamma}$)

<div style="
    background-color: #f8f9fa; 
    border-left: 5px solid #ced4da; 
    padding: 15px; 
    margin: 10px 0; 
    border-radius: 4px; 
    color: #495057;
">
    <strong>Reference:</strong> See Sec 4.3 in Geier et al. (2015): from Eq. (B.1) to (B.3).
</div>

<div style="
    background-color: #f0f8ff; 
    border-left: 5px solid #a2c4c9; 
    padding: 15px; 
    margin: 10px 0; 
    border-radius: 4px; 
    color: #2c4a5e;
">
    <strong>Note:</strong> The implementation here differs from that in Geier et al., in which the central moments are directly transformed from the distribution. See Eqs. (43)-(45) in the literature. 
</div>

Although `sp.cse` can calculations in the moment transformation efficient, we utilize the chimera fast moment transformation proposed by Geier et al. (2015) to make our implementation consistent: 

\begin{align}
&m_{ij|\gamma} = \sum_{k=-1,0,1} k^{\gamma} f_{ijk} \\
&m_{i|\beta \gamma} = \sum_{j=-1,0,1} j^{\beta} m_{ij|\gamma} \\
&m_{\alpha \beta \gamma}  = \sum_{i=-1,0,1} i^{\alpha} m_{i|\beta \gamma}
\end{align}

We develop a function creating *chimera*s: $m_{ij|\gamma} = \sum_{k=-1,0,1} k^{\gamma} f_{ijk}$ and $m_{i|\beta \gamma} = \sum_{j=-1,0,1} j^{\beta} m_{ij|\gamma}$

In [20]:
chimera_name = 'chimera'
chimera_delimiter = 'c'

def chimera_moment(dim, vectors, pipe_exprs=None):
    if pipe_exprs == None:
        pipe_exprs = {}
        
    # dynamic loop based on vectors
    # dim=3: grouping (v[0], v[1]) and product sum for v[2] (k)
    # dim=2: grouping v[0] and product sum for v[1] (j)
    sum_axis = dim - 1
    group_keys = sorted(list(set(v[:sum_axis] for v in vectors)))
    v_map = {-1: "m1", 0: "0", 1: "1"}
    
    first_chimera = {} # m_{i|beta gamma}
    
    for g_key in group_keys:
        for gamma in [0, 1, 2]:
            expr = 0
            for idx, v in enumerate(vectors):
                if v[:sum_axis] == g_key:
                    k_val = v[sum_axis]
                    k_pow = 1 if (k_val == 0 and gamma == 0) else (k_val ** gamma)
                    expr += k_pow * sp.Symbol(f"f{idx}") # m_{ij|gamma} = sum k^{gamma} f_{ijk}

            expr = sp.simplify(expr)
            if dim == 3:
                var_sym = sp.Symbol(f"{chimera_name}_m_{v_map[g_key[0]]}_{v_map[g_key[1]]}_{chimera_delimiter}_{gamma}")
            else:
                var_sym = sp.Symbol(f"{chimera_name}_m_{v_map[g_key[0]]}_{chimera_delimiter}_{gamma}")
                
            first_chimera[(g_key, gamma)] = var_sym
            
            pipe_exprs[var_sym] = expr
                        
    second_chimera = {} # m_{ij|gamma}

    if dim == 3:
        x_keys = sorted(list(set(v[0] for v in vectors)))
        for i_val in x_keys:
            for beta in [0, 1, 2]:
                for gamma in [0, 1, 2]:
                    expr = 0
                    for j_val in [-1, 0, 1]:
                        if ((i_val, j_val), gamma) in first_chimera:
                            j_pow = 1 if (j_val == 0 and beta == 0) else (j_val ** beta)
                            expr += j_pow * first_chimera[((i_val, j_val), gamma)] # m_{i|beta gamma} = sum j^{beta} m_{ij|gamma}  

                    expr = sp.simplify(expr)
                    var_sym = sp.Symbol(f"{chimera_name}_m_{v_map[i_val]}_{chimera_delimiter}_{beta}_{gamma}")
                    second_chimera[(i_val, beta, gamma)] = var_sym
                    
                    pipe_exprs[var_sym] = expr
    
    return first_chimera, second_chimera, v_map, pipe_exprs

In [21]:
def generate_moment_expr(o, first_chimera, second_chimera):
    dim = len(o) 
    
    expr_raw = 0
    
    if dim == 3:
        alpha, beta, gamma = o
        for i_val in [-1, 0, 1]:
            if (i_val, beta, gamma) in second_chimera:
                i_pow = 1 if (i_val == 0 and alpha == 0) else (i_val ** alpha)
                expr_raw += i_pow * second_chimera[(i_val, beta, gamma)] # m_{alpha beta gamma} = sum i^{alpha} m_{i|beta gamma}  

    else:  # dim == 2
        alpha, beta = o
        for i_val in [-1, 0, 1]:
            target_key = ((i_val,), beta)
            if target_key in first_chimera:
                i_pow = 1 if (i_val == 0 and alpha == 0) else (i_val ** alpha)
                expr_raw += i_pow * first_chimera[target_key]
                
    return sp.simplify(expr_raw)

In [22]:
first_chimera, second_chimera, v_map, pipe_exprs_m_chimera = chimera_moment(dim, vectors) 

The first chimera, $m_{ij|\gamma} = \sum_{k=-1,0,1} k^{\gamma} f_{ijk}$, has the suffix $(ij)$ and $\gamma$ on the L.H.S. This function represents this fact by using two args in accesser, meaning that, the argument is a tuple of *((i,j), gamma)*

In [23]:
print( first_chimera[(-1,-1)[:dim-1], 0] ) 

chimera_m_m1_m1_c_0


The second chimera, $m_{i|\beta \gamma} = \sum_{j=-1,0,1} j^{\beta} m_{ij|\gamma}$, has the suffix $i$ and $\beta \gamma$. Therefore, in this case, we employ a flat accesser *(i,beta,gamma)*: 

In [24]:
if dim == 3: # exists only in 3D
    print( second_chimera[(-1,0,0)] )

chimera_m_m1_c_0_0


The chimera expressions can be accessed via `pipe_exprs_m_chimera`

In [25]:
display( Eq(first_chimera[(-1,-1)[:dim-1], 0], pipe_exprs_m_chimera[first_chimera[(-1,-1)[:dim-1], 0]]) )
if dim == 3:
    display( Eq(second_chimera[(-1,0,0)], pipe_exprs_m_chimera[second_chimera[(-1,0,0)]]) )

Eq(chimera_m_m1_m1_c_0, f20 + f22 + f8)

Eq(chimera_m_m1_c_0_0, chimera_m_m1_0_c_0 + chimera_m_m1_1_c_0 + chimera_m_m1_m1_c_0)

Is this correct? Let's check by hand.

$m_{-1-1|0} = \sum_{k=-1,0,1} k^{0} f_{-1-1k} = \sum_{k=-1,0,1} f_{-1-1k} = f_{-1-1-1} + f_{-1-10} + f_{-1-11}$

The correctness of the suffix can be confirmed by observing the vector components, i.e.,

In [26]:
if dim == 3:
    display(vectors[20], vectors[22], vectors[8])

(-1, -1, -1)

(-1, -1, 1)

(-1, -1, 0)

Good! Then, the second chimera, 

$m_{-1|00} = \sum_{j=-1,0,1} j^{0} m_{-1j|0} = m_{-1-1|0} + m_{-10|0} + m_{-11|0}$

The raw moments are then constructed from the chimeras, e.g.,  

In [27]:
generate_moment_expr((1,1,1)[:dim], first_chimera, second_chimera)

chimera_m_1_c_1_1 - chimera_m_m1_c_1_1

Prepare moment name *dict* for convenience in constructing equations.

In [28]:
def create_moment_dictionary(moment_orders, rho=None):
    M_raw, M_post = {}, {} # raw moment
    K_cen, K_post = {}, {} # central moment 
    C_cum, C_post = {}, {} # cumulant (density scaled) 

    for o in moment_orders:
        o_name = "".join(map(str, o))
        order_sum = sum(o)
        
        M_raw[o]  = sp.Symbol(f"m{o_name}")
        M_post[o] = sp.Symbol(f"m{o_name}_post")

        K_cen[o]  = sp.Symbol(f"kappa{o_name}")
        K_post[o] = sp.Symbol(f"kappa{o_name}_post")
        
        C_cum[o]  = sp.Symbol(f"C{o_name}")
        C_post[o] = sp.Symbol(f"C{o_name}_post")
        
    return M_raw, M_post, K_cen, K_post, C_cum, C_post

In [29]:
M_raw, M_post, K_cen, K_post, C_cum, C_post = create_moment_dictionary(moment_orders)

Set *dict* of all raw moment expressions:

In [30]:
M_raw_expr = {}

for o, name in zip(moment_orders, mom_names):
    M_raw_expr[o] = generate_moment_expr(o, first_chimera, second_chimera)

In [31]:
for o, name in zip(moment_orders, mom_names):
    print(f"{M_raw[o]} = {M_raw_expr[o]}")

m000 = chimera_m_0_c_0_0 + chimera_m_1_c_0_0 + chimera_m_m1_c_0_0
m100 = chimera_m_1_c_0_0 - chimera_m_m1_c_0_0
m001 = chimera_m_0_c_0_1 + chimera_m_1_c_0_1 + chimera_m_m1_c_0_1
m010 = chimera_m_0_c_1_0 + chimera_m_1_c_1_0 + chimera_m_m1_c_1_0
m200 = chimera_m_1_c_0_0 + chimera_m_m1_c_0_0
m101 = chimera_m_1_c_0_1 - chimera_m_m1_c_0_1
m110 = chimera_m_1_c_1_0 - chimera_m_m1_c_1_0
m002 = chimera_m_0_c_0_2 + chimera_m_1_c_0_2 + chimera_m_m1_c_0_2
m011 = chimera_m_0_c_1_1 + chimera_m_1_c_1_1 + chimera_m_m1_c_1_1
m020 = chimera_m_0_c_2_0 + chimera_m_1_c_2_0 + chimera_m_m1_c_2_0
m201 = chimera_m_1_c_0_1 + chimera_m_m1_c_0_1
m210 = chimera_m_1_c_1_0 + chimera_m_m1_c_1_0
m102 = chimera_m_1_c_0_2 - chimera_m_m1_c_0_2
m111 = chimera_m_1_c_1_1 - chimera_m_m1_c_1_1
m120 = chimera_m_1_c_2_0 - chimera_m_m1_c_2_0
m012 = chimera_m_0_c_1_2 + chimera_m_1_c_1_2 + chimera_m_m1_c_1_2
m021 = chimera_m_0_c_2_1 + chimera_m_1_c_2_1 + chimera_m_m1_c_2_1
m202 = chimera_m_1_c_0_2 + chimera_m_m1_c_0_2
m211 = chime

## Forward transformation: $m_{\alpha \beta \gamma} \rightarrow \kappa_{\alpha \beta \gamma}$

The central moment is derived from the raw moments by definition: 

\begin{equation}
\kappa_{\alpha \beta \gamma} = \sum_{i=0}^{Q-1} (c_{ix}-u_{x})^{\alpha} (c_{iy}-u_{y})^{\beta} (c_{iz}-u_{z})^{\gamma} f_{i}
\end{equation}

For example, 

\begin{align}
    \kappa_{111} 
    &= \sum_{i=0}^{Q-1} (c_{ix}-u_{x})^{1} (c_{iy}-u_{y})^{1} (c_{iz}-u_{z})^{1} f_{i} \\
    &= \sum_{i=0}^{Q-1} ( c_{ix}c_{iy}c_{iz} - u_{x}c_{iy}c_{iz} - u_{z}c_{ix}c_{iy} + u_{x}u_{z}c_{iy} - u_{y}c_{ix}c_{iz} + u_{x}u_{y} c_{iz} + u_{y}u_{z}c_{ix} - u_{x}u_{y}u_{z} ) f_{i} \\
   &= m_{000} - u m_{011} - v m_{101} - w m_{110} + 2 \rho u v w \\
\end{align}


In [32]:
def replace_with_moments(expr, c_list, prefix='m'): # X**2 * Z**2 -> m202
    poly = sp.Poly(expr, c_list)
    terms = poly.terms()

    new_expr = 0
    
    for exponent_tuple, coeff in terms:
        suffix = "".join(map(str, exponent_tuple))
        sym = sp.Symbol(f"{prefix}{suffix}")
        new_expr += coeff * sym

    return new_expr


def derive_central_moment_from_moment(order, rho, u):
    dim = len(order)

    c_list = sp.symbols( ['cx', 'cy', 'cz'][:dim] ) # lattice velocity

    m000 = sp.Symbol('m' + '0'*dim)

    kappa_expr = sp.prod([(c_list[i] - u[i])**order[i] for i in range(dim)]) # construct kappa by definition
    kappa_expr = sp.expand(kappa_expr)
    kappa_expr = replace_with_moments(kappa_expr, c_list, prefix='m')

    u_subs = {u[i]: sp.Symbol(f'm{"0"*i}1{"0"*(dim-1-i)}') / m000 for i in range(dim)}
    kappa_expr = kappa_expr.xreplace(u_subs)

    back_subs = {m000: rho} # replace m000 with rho

    for i in range(dim): # replace m100, m010, m001 with rho u, rho v, rho w
        mom_name = f'm{"0"*i}1{"0"*(dim-1-i)}'
        
        back_subs[sp.Symbol(mom_name)] = rho * u[i]
    
    kappa_expr = kappa_expr.xreplace(back_subs)

    lhs = sp.Symbol(f'\\kappa_{{{"".join(map(str, order))}}}')

    return Eq(lhs, kappa_expr)

In [33]:
for o, name in zip(moment_orders, mom_names):
    display( derive_central_moment_from_moment(o, rho, vel_syms) )

Eq(\kappa_{000}, rho)

Eq(\kappa_{100}, 0)

Eq(\kappa_{001}, 0)

Eq(\kappa_{010}, 0)

Eq(\kappa_{200}, m200 - rho*u**2)

...

Eq(\kappa_{022}, m002*v**2 + 4*m011*v*w - 2*m012*v + m020*w**2 - 2*m021*w + m022 - 3*rho*v**2*w**2)

Eq(\kappa_{212}, -m002*u**2*v - 2*m011*u**2*w + m012*u**2 - 4*m101*u*v*w + 2*m102*u*v - 2*m110*u*w**2 + 4*m111*u*w - 2*m112*u - m200*v*w**2 + 2*m201*v*w - m202*v + m210*w**2 - 2*m211*w + m212 + 4*rho*u**2*v*w**2)

Eq(\kappa_{221}, -2*m011*u**2*v - m020*u**2*w + m021*u**2 - 2*m101*u*v**2 - 4*m110*u*v*w + 4*m111*u*v + 2*m120*u*w - 2*m121*u - m200*v**2*w + m201*v**2 + 2*m210*v*w - 2*m211*v - m220*w + m221 + 4*rho*u**2*v**2*w)

Eq(\kappa_{122}, -m002*u*v**2 - 4*m011*u*v*w + 2*m012*u*v - m020*u*w**2 + 2*m021*u*w - m022*u - 2*m101*v**2*w + m102*v**2 - 2*m110*v*w**2 + 4*m111*v*w - 2*m112*v + m120*w**2 - 2*m121*w + m122 + 4*rho*u*v**2*w**2)

Eq(\kappa_{222}, m002*u**2*v**2 + 4*m011*u**2*v*w - 2*m012*u**2*v + m020*u**2*w**2 - 2*m021*u**2*w + m022*u**2 + 4*m101*u*v**2*w - 2*m102*u*v**2 + 4*m110*u*v*w**2 - 8*m111*u*v*w + 4*m112*u*v - 2*m120*u*w**2 + 4*m121*u*w - 2*m122*u + m200*v**2*w**2 - 2*m201*v**2*w + m202*v**2 - 2*m210*v*w**2 + 4*m211*v*w - 2*m212*v + m220*w**2 - 2*m221*w + m222 - 5*rho*u**2*v**2*w**2)

These are $\kappa$ as functions of $m$. 

We, for higher than 2nd order, express them in terms of *the raw moment* $m$ of same order and *central moemnts* $\kappa$ of lower orders by a cascading process. 

In [34]:
K_cen_expr = {}

kappa_subs_dict = {}

for o, name in zip(moment_orders, mom_names):

    kappa_eq = derive_central_moment_from_moment(o, rho, vel_syms) # get kappa in terms of raw moments from generating fucntion
    
    k_sym, expr = kappa_eq.lhs, kappa_eq.rhs
    K_cen_expr[o] = expr # register kappa expr

    if sum(o) == 2:
        kappa_subs_dict.update({name: sp.solve(kappa_eq, name)[0]}) # solver for raw moment and add to subs_dict 

    if sum(o) > 2:
        K_cen_expr[o] = sp.simplify(K_cen_expr[o].subs(kappa_subs_dict)) # replace raw moments with lower order central moments
        kappa_subs_dict.update({k_sym: K_cen_expr[o]}) # add kappa expression to subs_dict for replacement at the next order
        kappa_subs_dict.update({name: sp.solve( Eq(k_sym, K_cen_expr[o]), name)[0]}) # solve for raw moment of the same order and add to sub_dict for replacement at the next order

    if dim == 2:
        if sum(o) == 4:
            display( Eq(k_sym, K_cen_expr[o]) )
    else: # dim = 3
        if sum(o) > 4:
            display( Eq(k_sym, K_cen_expr[o]) )
        
        # some examples at high orders are shown below
        

Eq(\kappa_{212}, -\kappa_{002}*u**2*v - 2*\kappa_{011}*u**2*w - \kappa_{012}*u**2 - 4*\kappa_{101}*u*v*w - 2*\kappa_{102}*u*v - 2*\kappa_{110}*u*w**2 - 4*\kappa_{111}*u*w - 2*\kappa_{112}*u - \kappa_{200}*v*w**2 - 2*\kappa_{201}*v*w - \kappa_{202}*v - \kappa_{210}*w**2 - 2*\kappa_{211}*w + m212 - rho*u**2*v*w**2)

Eq(\kappa_{221}, -2*\kappa_{011}*u**2*v - \kappa_{020}*u**2*w - \kappa_{021}*u**2 - 2*\kappa_{101}*u*v**2 - 4*\kappa_{110}*u*v*w - 4*\kappa_{111}*u*v - 2*\kappa_{120}*u*w - 2*\kappa_{121}*u - \kappa_{200}*v**2*w - \kappa_{201}*v**2 - 2*\kappa_{210}*v*w - 2*\kappa_{211}*v - \kappa_{220}*w + m221 - rho*u**2*v**2*w)

Eq(\kappa_{122}, -\kappa_{002}*u*v**2 - 4*\kappa_{011}*u*v*w - 2*\kappa_{012}*u*v - \kappa_{020}*u*w**2 - 2*\kappa_{021}*u*w - \kappa_{022}*u - 2*\kappa_{101}*v**2*w - \kappa_{102}*v**2 - 2*\kappa_{110}*v*w**2 - 4*\kappa_{111}*v*w - 2*\kappa_{112}*v - \kappa_{120}*w**2 - 2*\kappa_{121}*w + m122 - rho*u*v**2*w**2)

Eq(\kappa_{222}, -\kappa_{002}*u**2*v**2 - 4*\kappa_{011}*u**2*v*w - 2*\kappa_{012}*u**2*v - \kappa_{020}*u**2*w**2 - 2*\kappa_{021}*u**2*w - \kappa_{022}*u**2 - 4*\kappa_{101}*u*v**2*w - 2*\kappa_{102}*u*v**2 - 4*\kappa_{110}*u*v*w**2 - 8*\kappa_{111}*u*v*w - 4*\kappa_{112}*u*v - 2*\kappa_{120}*u*w**2 - 4*\kappa_{121}*u*w - 2*\kappa_{122}*u - \kappa_{200}*v**2*w**2 - 2*\kappa_{201}*v**2*w - \kappa_{202}*v**2 - 2*\kappa_{210}*v*w**2 - 4*\kappa_{211}*v*w - 2*\kappa_{212}*v - \kappa_{220}*w**2 - 2*\kappa_{221}*w + m222 - rho*u**2*v**2*w**2)

## Forward transformation: $\kappa_{\alpha \beta \gamma} \rightarrow C_{\alpha \beta \gamma}$

The cumulant generating function is defined by 
\begin{equation}
    C ( \mathbf{X} ) = \log M( \mathbf{X} ) 
\end{equation}
while it can also be written in terms of the central moment generating function $K$: 
\begin{equation}
    C ( \mathbf{X} ) = \mathbf{X} \cdot \mathbf{u} + \log K( \mathbf{X} )
\end{equation}
The cumulants are  
\begin{equation}
    c_{\alpha \beta \gamma} = \left[ \partial_{X}^{\alpha} \partial_{Y}^{\beta} \partial_{Z}^{\gamma} C( \mathbf{X} ) \right]_{\mathbf{X}=0}
\end{equation}
Hence, the relationships between cumulants and central moments required in the collision step can be derived by differentiating the generating function. See [lbmpy Tutorial 04](https://pycodegen.pages.i10git.cs.fau.de/lbmpy/notebooks/04_tutorial_cumulant_LBM.html). 

Here, we develop a cumulant generator class to obtain cumulants in terms of $\kappa$. 

<div style="
    background-color: #f0f8ff; 
    border-left: 5px solid #a2c4c9; 
    padding: 15px; 
    margin: 10px 0; 
    border-radius: 4px; 
    color: #2c4a5e;
">
    <strong>Note:</strong> This class was originally desinged for `cumulant_moment_exprs.ipynb`. Some symbols defined with "_{}" to represent its order. This requires string replacement in the following calculation. You can, of course, modify the code!
</div>

In [35]:
class CumulantGenerator():
    def __init__(self, dim=3):
        self.dim = dim
        self.X_list = sp.symbols(['X', 'Y', 'Z'][:dim])
        self.u_list = sp.symbols(['u', 'v', 'w'][:dim])
        self.K = sp.Function('K')(*(self.X_list))
        self.C = sp.Function('C')(*(self.X_list))
        self.X_vec = sp.Matrix(self.X_list)
        self.u_vec = sp.Matrix(self.u_list)
        self.C_sym = (self.X_vec.T @ self.u_vec)[0] + sp.log(self.K)

        self.alpha, self.beta, self.gamma = sp.symbols('alpha beta gamma', integer=True, positive=True)
        self.order_sym = (self.alpha, self.beta, self.gamma)
        diff_vars = []
        for var, n in zip(self.X_list, self.order_sym):
            diff_vars.append((var, n))

        self.gen_operator = sp.Derivative(self.C, *diff_vars)
        self.density_scaling = True

    def definition_of_generating_function(self):
        return Eq(self.C, self.C_sym)

    def definition_of_cumulant(self):
        lhs_def = sp.Symbol(rf'c_{{\alpha \beta \gamma}}', latex_name=rf'c_{{\alpha \beta \gamma}}')
        return Eq(lhs_def, self.gen_operator)

    def __call__(self, order, density_scaling=None):
        if density_scaling == None: density_scaling = self.density_scaling
        c_expr = self._generate_cumulant_from_central_moment(order, density_scaling=density_scaling)
        lhs = sp.Symbol(f'C_{{{"".join(map(str, order))}}}') if density_scaling else sp.Symbol(f'c_{{{"".join(map(str, order))}}}')
        return Eq(lhs, c_expr)

    def _generate_cumulant_from_central_moment(self, order_tuple, density_scaling):
        dim = self.dim
        if self.dim != len(order_tuple): 
            raise ValueError(f"Dimension mismatch: expected {self.dim}, got {len(order_tuple)}.")

        rho = sp.Symbol('rho') # density

        expr = self.C_sym # cumulant generating function in terms of K
        for var, order in zip(self.X_list, order_tuple):
            if order > 0:
                expr = expr.diff(var, order)
        
        def get_deriv_symbol(deriv_obj):
            # enforce kappa100=kappa010=kappa001=0 to suppress term divergence at high orders
            args = deriv_obj.args # get derivative order in Derivative expr, like (X, 2) for partial_{X}^{2} 

            orders = {var: 0 for var in self.X_list} # derivative order count; starting from X:0 Y:0 Z:0
            for arg in args[1:]:
                var, count = arg if isinstance(arg, (tuple, sp.Tuple)) else (arg, 1)            
                if var in orders:
                    orders[var] += count

            total = sum(orders.values())
            
            if total == 0: # 000
                return rho
            elif total == 1: # kappa100 -> 0
                return sp.Integer(0)
            else:
                suffix = "".join([str(orders[var]) for var in self.X_list])
                return sp.Symbol(f'kappa{suffix}')

        subs_rules = {d: get_deriv_symbol(d) for d in expr.atoms(sp.Derivative)} # dictionary for replacement of derivatives of K with kappa symbols        
        subs_rules.update({var: 0 for var in self.X_list}) # evaluate at X=0 (treatment for X dot u term)
        subs_rules[self.K] = rho # replacement rule: K at X=0 -> rho
        multiplyer = rho if density_scaling==True else sp.Integer(1) # density scaling C_{200} = rho c_{200} = kappa_{200}

        return ( expr.subs(subs_rules)*multiplyer ).simplify()

Instantiate CumulantGenerator object as `Cgen`.

In [36]:
Cgen = CumulantGenerator(dim=dim)

Using `Cgen`, generate cumulant expressions.

<div style="
    background-color: #f0f8ff; 
    border-left: 5px solid #a2c4c9; 
    padding: 15px; 
    margin: 10px 0; 
    border-radius: 4px; 
    color: #2c4a5e;
">
    <strong>Note:</strong> We will use only $C_{\alpha \beta \gamma}$ higher than 3rd orders since the 2nd and 3rd order cumulants are the same as $\kappa$. However, we generate all $C_{\alpha \beta \gamma}$ just for confirmation. 
</div>

In [37]:
C_cum_expr = {}

for i in range(len(moment_orders)):
    o, name = moment_orders[i], mom_names[i]

    C_eq = Cgen(o, density_scaling=True) # get cumulant
    
    C_cum_expr[o] = C_eq.rhs # register cumulant expr

    display( Eq(C_eq.lhs, C_cum_expr[o]) )

Eq(C_{000}, rho*log(rho))

Eq(C_{100}, rho*u)

Eq(C_{001}, rho*w)

Eq(C_{010}, rho*v)

Eq(C_{200}, kappa200)

...

Eq(C_{022}, (-kappa002*kappa020 - 2*kappa011**2 + kappa022*rho)/rho)

Eq(C_{212}, (-kappa002*kappa210 - 2*kappa011*kappa201 - kappa012*kappa200 - 4*kappa101*kappa111 - 2*kappa102*kappa110 + kappa212*rho)/rho)

Eq(C_{221}, (-2*kappa011*kappa210 - kappa020*kappa201 - kappa021*kappa200 - 2*kappa101*kappa120 - 4*kappa110*kappa111 + kappa221*rho)/rho)

Eq(C_{122}, (-kappa002*kappa120 - 4*kappa011*kappa111 - 2*kappa012*kappa110 - kappa020*kappa102 - 2*kappa021*kappa101 + kappa122*rho)/rho)

Eq(C_{222}, (2*kappa002*kappa020*kappa200 + 4*kappa002*kappa110**2 - kappa002*kappa220*rho + 4*kappa011**2*kappa200 + 16*kappa011*kappa101*kappa110 - 4*kappa011*kappa211*rho - 2*kappa012*kappa210*rho + 4*kappa020*kappa101**2 - kappa020*kappa202*rho - 2*kappa021*kappa201*rho - kappa022*kappa200*rho - 4*kappa101*kappa121*rho - 2*kappa102*kappa120*rho - 4*kappa110*kappa112*rho - 4*kappa111**2*rho + kappa222*rho**2)/rho**2)

Now, we have finished the forward transformation! 

Close the forward pipeline. 

In [38]:
pipe_forward = pipe_exprs_m_chimera

for o in moment_orders:
    pipe_forward[M_raw[o]] = M_raw_expr[o]

for o in moment_orders:
    if sum(o) > 1:
        pipe_forward[K_cen[o]] = K_cen_expr[o]

for o in moment_orders:
    if sum(o) > 3:
        pipe_forward[C_cum[o]] = C_cum_expr[o]

## Collision and backward transformation $C_{\alpha \beta \gamma} \rightarrow \kappa_{\alpha \beta \gamma}$

<div style="
    background-color: #f8f9fa; 
    border-left: 5px solid #ced4da; 
    padding: 15px; 
    margin: 10px 0; 
    border-radius: 4px; 
    color: #495057;
">
    <strong>Reference:</strong> See Sec 4.3 in Geier et al. (2015): from Eq. (55) to (84).
</div>

Collision step proceeds in the cumulant space. However, as we confirmed above, the 2nd and 3rd-order cumulants are the same as central moments. 

In [39]:
num_params = 10 # [10 omegas]
om_syms = [None] + list(sp.symbols(f'om1:{num_params + 1}')) # sp.symbols('om1 om2 ... om10') # om_syms[1] -> om1

omega_values = [f"omega[{i}]" for i in range(1, num_params + 1)]
omega_config = {}
for i in range(1, len(om_syms)):
    val = omega_values[i-1] # note: 0-indexed, so shift i
    omega_config[om_syms[i]] = sp.Symbol(val)            

We prepare order tuples for each order. This may be useful since we need to do *for* loops order by order.  

In [40]:
one = sp.Integer(1)

orders_2nd = [o for o in moment_orders if sum(o) == 2]
orders_3rd = [o for o in moment_orders if sum(o) == 3]
orders_4th = [o for o in moment_orders if sum(o) == 4]
orders_5th = [o for o in moment_orders if sum(o) == 5]
orders_6th = [o for o in moment_orders if sum(o) == 6]

We follow the collision equations given by Geier et al. (2015). The equations are registered into a pipeline: `pipe_collision` *dict*. 

In [41]:
pipe_collision = {}

We begin by low order cumulants. The first ones are the second order cumulants $C_{110}$, $C_{101}$, $C_{011}$. Their equilibirum $C^{eq}$ is $0$. Therefore, 

\begin{align}
&C_{110}^{*} = (1 - \omega_{1}) C_{110} \\
&C_{101}^{*} = (1 - \omega_{1}) C_{101} \\
&C_{011}^{*} = (1 - \omega_{1}) C_{011} 
\end{align}

In [42]:
cross_orders_2nd = [o for o in orders_2nd if max(o) == 1] # 200 is also 2nd order, but considered later with different logic

for o in cross_orders_2nd:
    relaxation_expr = K_cen[o] + omega_config[om_syms[1]] * (0 - K_cen[o]) # make expr
    pipe_collision[K_post[o]] = sp.simplify(relaxation_expr)               # then register it in pipeline

We write here `K_post`, not `C_post`. Up to the 3rd order, $C = \kappa$; therefore, writing $\kappa^{*}$ means that we have already transformed $C^{*}$ to $\kappa^{*}$!

`pipe_collision` looks like 

In [43]:
pipe_collision # check the equations registered!

{kappa101_post: kappa101*(1 - omega[1]),
 kappa110_post: kappa110*(1 - omega[1]),
 kappa011_post: kappa011*(1 - omega[1])}

The remaining 2nd order cumulants are $C_{200}$, $C_{020}$, $C_{002}$. Their collision equations are 
\begin{align}
&C_{200}^{*} + C_{020}^{*} - 2 C_{002}^{*} = (1 - \omega_{1}) ( C_{200} + C_{020} - 2 C_{002} ) \\
&C_{200}^{*} - C_{002}^{*} = (1 - \omega_{1}) ( C_{200} - C_{002} ) \\
&C_{200}^{*} + C_{020}^{*} + C_{002}^{*} = \operatorname{Tr}(P) + (1 - \omega_{2}) ( C_{200} + C_{020} + C_{002} )
\end{align}
where
$P_{ij} = p \delta_{ij} = (\rho/3) \delta_{ij}$; $\operatorname{Tr}(P) = \rho$ in 3D, $2\rho/3$ in 2D. Let the right-hand sides be $X$, $Y$ and $Z$, the post-collision cumulants are

In [44]:
C200, C020, C002, X, Y, Z = sp.symbols('C200 C020 C002 X Y Z')

eq1 = C200 + C020 - 2*C002 - X
eq2 = C200 - C020 - Y
eq3 = C200 + C020 + C002 - Z

sol = sp.solve([eq1, eq2, eq3], [C200, C020, C002])

for key, val in sol.items():
    display( Eq(key, sp.factor(val)) )

Eq(C002, -(X - Z)/3)

Eq(C020, (X - 3*Y + 2*Z)/6)

Eq(C200, (X + 3*Y + 2*Z)/6)

We set symbolic equations and push them into the pipeline. 

In [45]:
kappa_diag_eq = sp.Rational(1, 3) * rho # isotropic component (pressure): rho * cs**2 = rho / 3

if dim == 2:
    trace_mode = K_cen[(2, 0)] + K_cen[(0, 2)]
    trace_eq   = 2 * kappa_diag_eq # trace of eqilibrium values: 2 * rho / 3
    trace_post = trace_mode + omega_config[om_syms[2]] * (trace_eq - trace_mode)
    
    diff_mode  = K_cen[(2, 0)] - K_cen[(0, 2)]
    diff_post  = diff_mode + omega_config[om_syms[1]] * (0 - diff_mode)
    
    pipe_collision[K_post[(2, 0)]] = sp.simplify((trace_post + diff_post) * sp.Rational(1, 2))
    pipe_collision[K_post[(0, 2)]] = sp.simplify((trace_post - diff_post) * sp.Rational(1, 2))
else:
    trace_mode = K_cen[(2,0,0)] + K_cen[(0,2,0)] + K_cen[(0,0,2)]
    trace_eq   = 3 * kappa_diag_eq # trace of eqilibrium values: 3 * (rho/3) = rho
    trace_post = trace_mode + omega_config[om_syms[2]] * (trace_eq - trace_mode)

    diff1_mode = (K_cen[(2,0,0)] - K_cen[(0,2,0)])
    diff1_post = diff1_mode + omega_config[om_syms[1]] * (0 - diff1_mode)        
    diff2_mode = (K_cen[(2,0,0)] + K_cen[(0,2,0)] - 2 * K_cen[(0,0,2)])
    diff2_post = diff2_mode + omega_config[om_syms[1]] * (0 - diff2_mode)

    pipe_collision[K_post[(2,0,0)]] = sp.simplify((2 * trace_post + 3 * diff1_post + diff2_post) * sp.Rational(1, 6))
    pipe_collision[K_post[(0,2,0)]] = sp.simplify((2 * trace_post - 3 * diff1_post + diff2_post) * sp.Rational(1, 6))
    pipe_collision[K_post[(0,0,2)]] = sp.simplify((trace_post - diff2_post) * sp.Rational(1, 3))


We have 7 at the 3rd order: 
$C_{120}$, $C_{102}$, $C_{210}$, $C_{012}$, $C_{201}$, $C_{021}$, $C_{111}$

$C_{111}^{*}$ is simply given by 

$C_{111}^{*} = (1 - \omega_{5}) C_{111}$ 

where $C_{111}^{eq} = 0$. The others are 

\begin{align}
&C_{120}^{*} + C_{102}^{*} = (1 - \omega_{3}) (C_{120} + C_{102}) \\
&C_{210}^{*} + C_{012}^{*} = (1 - \omega_{3}) (C_{210} + C_{012}) \\
&C_{201}^{*} + C_{021}^{*} = (1 - \omega_{3}) (C_{201} + C_{021}) \\
&C_{120}^{*} - C_{102}^{*} = (1 - \omega_{4}) (C_{120} - C_{102}) \\
&C_{210}^{*} - C_{012}^{*} = (1 - \omega_{4}) (C_{210} - C_{012}) \\
&C_{201}^{*} - C_{021}^{*} = (1 - \omega_{4}) (C_{201} - C_{021}) \\
\end{align}
We can easily solve for each, for example, 

$C_{120}^{*} = (1 - \omega_{3}) (C_{120} + C_{102})/2 +  (1 - \omega_{4}) (C_{120} - C_{102})/2$


In [46]:
if dim == 2:
    for o in orders_3rd: # goes to zero equilibria
        relaxation_expr = K_cen[o] + omega_config[om_syms[3]] * (sp.Integer(0) - K_cen[o])
        pipe_collision[K_post[o]] = sp.simplify(relaxation_expr)
else: 
    # 120, 102, 210, 012, 201, 021
    sum1_post  = (one - omega_config[om_syms[3]]) * (K_cen[(1,2,0)] + K_cen[(1,0,2)])
    sum2_post  = (one - omega_config[om_syms[3]]) * (K_cen[(2,1,0)] + K_cen[(0,1,2)])
    sum3_post  = (one - omega_config[om_syms[3]]) * (K_cen[(2,0,1)] + K_cen[(0,2,1)])
    diff1_post = (one - omega_config[om_syms[4]]) * (K_cen[(1,2,0)] - K_cen[(1,0,2)])
    diff2_post = (one - omega_config[om_syms[4]]) * (K_cen[(2,1,0)] - K_cen[(0,1,2)])
    diff3_post = (one - omega_config[om_syms[4]]) * (K_cen[(2,0,1)] - K_cen[(0,2,1)])
    pipe_collision[K_post[(1,2,0)]] = sp.simplify((sum1_post + diff1_post) * sp.Rational(1, 2))
    pipe_collision[K_post[(2,1,0)]] = sp.simplify((sum2_post + diff2_post) * sp.Rational(1, 2))
    pipe_collision[K_post[(2,0,1)]] = sp.simplify((sum3_post + diff3_post) * sp.Rational(1, 2))
    pipe_collision[K_post[(1,0,2)]] = sp.simplify((sum1_post - diff1_post) * sp.Rational(1, 2))
    pipe_collision[K_post[(0,1,2)]] = sp.simplify((sum2_post - diff2_post) * sp.Rational(1, 2))
    pipe_collision[K_post[(0,2,1)]] = sp.simplify((sum3_post - diff3_post) * sp.Rational(1, 2))

    # 111 
    o = (1,1,1)
    pipe_collision[K_post[o]] = sp.simplify( (one - omega_config[om_syms[5]]) * K_cen[o] )


We have actually done not only collision but also $C \rightarrow \kappa$ transformation at the 3rd order. 

We shall go for the higher orders. By using `Cgen`, a cumulant expression in terms of $\kappa$ can be easily obtained, e.g, 

In [47]:
o = (2,2,0)[:dim]
CO = Cgen(o, density_scaling=True)
display(CO)

Eq(C_{220}, (-kappa020*kappa200 - 2*kappa110**2 + kappa220*rho)/rho)

We notice that the post-collision moment $\kappa^{*}$ of the lower orders were already obtained. Therefore, after computing $C_{220}^{*}$, $\kappa_{220}^{*}$ can be calculated by solving the above equation for $\kappa_{220}$, that is, 

In [48]:
kappaO_post = sp.simplify(sp.solve(CO, K_cen[o])[0])
display( Eq(K_cen[o], kappaO_post) )

Eq(kappa220, (C_{220}*rho + kappa020*kappa200 + 2*kappa110**2)/rho)

However, we do not have *post* decorator in this equation derived with `Cgen`. So, prepare a simple label substitution map. 

In [49]:
k_post_subs_map = {}
for o in moment_orders:
    if sum(o) > 1:
        k_post_subs_map.update({K_cen[o]: sp.Symbol(str(K_cen[o])+'_post')})
        order = ''.join(str(x) for x in o)
        C_name = "C_{" + order + "}"
        k_post_subs_map.update({C_name: sp.Symbol(str(C_cum[o])+'_post')})

In [50]:
display( Eq(K_post[(2,2,0)[:dim]], kappaO_post.subs(k_post_subs_map)) )

Eq(kappa220_post, (C220_post*rho + kappa020_post*kappa200_post + 2*kappa110_post**2)/rho)

Very good. 

An elaborative part is only the fourth order of $C_{220}$ $C_{202}$ $C_{022}$. The collision operator for them is 

\begin{align}
&C_{220}^{*} - 2 C_{202}^{*} + C_{022}^{*} = (1 - \omega_{6}) (C_{220} - 2 C_{202} + C_{022}) \\ 
&C_{220}^{*} + C_{202}^{*} - 2 C_{022}^{*} = (1 - \omega_{6}) (C_{220} + C_{202} - 2 C_{022}) \\ 
&C_{220}^{*} + C_{202}^{*} + C_{022}^{*} = (1 - \omega_{7}) (C_{220} + C_{202} + C_{022}) \\ 
\end{align}

In [51]:
C220, C202, C022, X, Y, Z = sp.symbols('C220 C202 C022 X Y Z')

eq1 = C220 - 2*C202 + C022 - X
eq2 = C220 + C202 - 2*C022 - Y
eq3 = C220 + C202 + C022 - Z

sol = sp.solve([eq1, eq2, eq3], [C220, C202, C022])

for key, val in sol.items():
    display( Eq(key, sp.factor(val)) )

Eq(C022, -(Y - Z)/3)

Eq(C202, -(X - Z)/3)

Eq(C220, (X + Y + Z)/3)

In [52]:
special_4th_cross = {(2, 2, 0), (2, 0, 2), (0, 2, 2)}

if dim == 3:
    cross1_post = (C_cum[(2,2,0)] - 2 * C_cum[(2,0,2)] + C_cum[(0,2,2)]) * (one - omega_config[om_syms[6]])
    cross2_post = (C_cum[(2,2,0)] + C_cum[(2,0,2)] - 2 * C_cum[(0,2,2)]) * (one - omega_config[om_syms[6]])
    cross3_post = (C_cum[(2,2,0)] + C_cum[(2,0,2)] + C_cum[(0,2,2)]) * (one - omega_config[om_syms[7]])
    pipe_collision[C_post[(2,2,0)]] = sp.simplify( (   cross1_post + cross2_post + cross3_post ) * sp.Rational(1, 3) )
    pipe_collision[C_post[(2,0,2)]] = sp.simplify( ( - cross1_post               + cross3_post ) * sp.Rational(1, 3) )
    pipe_collision[C_post[(0,2,2)]] = sp.simplify( ( - cross2_post               + cross3_post ) * sp.Rational(1, 3) )

    for o in special_4th_cross:
        C_expr = Cgen(o, density_scaling=True)
        k_post_expr = sp.simplify(sp.solve(C_expr, K_cen[o])[0])
        pipe_collision[K_post[o]] = sp.expand(k_post_expr).subs(k_post_subs_map)


The last ones!

\begin{align}
&C_{211}^{*} = (1 - \omega_{8}) C_{211} \\
&C_{121}^{*} = (1 - \omega_{8}) C_{121} \\
&C_{112}^{*} = (1 - \omega_{8}) C_{112} \\
&C_{221}^{*} = (1 - \omega_{9}) C_{221} \\
&C_{212}^{*} = (1 - \omega_{9}) C_{212} \\
&C_{122}^{*} = (1 - \omega_{9}) C_{122} \\
&C_{222}^{*} = (1 - \omega_{10}) C_{222} \\
\end{align}

In [53]:
# cumulant transformation for orders higher than 3
all_higher_orders = orders_4th + orders_5th + orders_6th

processed = False
for o in all_higher_orders:
    if dim == 2: # (2, 2) is the only case for 2D, and C22_eq = 0
        if processed:
            pass
        else:
            pipe_collision[C_post[o]] = sp.simplify( (one - omega_config[om_syms[6]]) * C_cum[o] )

            C_expr = Cgen(o, density_scaling=True)
            k_post_expr = sp.simplify(sp.solve(C_expr, K_cen[o])[0])
            pipe_collision[K_post[o]] = sp.expand(k_post_expr).subs(k_post_subs_map)
        continue

    if dim == 3 and o in special_4th_cross:
        continue

    current_omega = omega_config[om_syms[8]]
    if sum(o) == 5: current_omega = omega_config[om_syms[9]]
    elif sum(o) == 6: current_omega = omega_config[om_syms[10]]

    pipe_collision[C_post[o]] = sp.simplify( (one - current_omega) * C_cum[o] )

    C_expr = Cgen(o, density_scaling=True)
    k_post_expr = sp.simplify(sp.solve(C_expr, K_cen[o])[0])
    pipe_collision[K_post[o]] = sp.expand(k_post_expr).subs(k_post_subs_map)


Let's see the outcome. 

In [54]:
for key, expr in pipe_collision.items():
    display( Eq(key, expr) )

Eq(kappa101_post, kappa101*(1 - omega[1]))

Eq(kappa110_post, kappa110*(1 - omega[1]))

Eq(kappa011_post, kappa011*(1 - omega[1]))

Eq(kappa200_post, kappa200 + omega[1]*(kappa020 - kappa200)/2 - omega[1]*(-2*kappa002 + kappa020 + kappa200)/6 - omega[2]*(kappa002 + kappa020 + kappa200 - rho)/3)

Eq(kappa020_post, kappa020 - omega[1]*(kappa020 - kappa200)/2 - omega[1]*(-2*kappa002 + kappa020 + kappa200)/6 - omega[2]*(kappa002 + kappa020 + kappa200 - rho)/3)

...

Eq(kappa221_post, C221_post + 2*kappa011_post*kappa210_post/rho + kappa020_post*kappa201_post/rho + kappa021_post*kappa200_post/rho + 2*kappa101_post*kappa120_post/rho + 4*kappa110_post*kappa111_post/rho)

Eq(C122_post, C122*(1 - omega[9]))

Eq(kappa122_post, C122_post + kappa002_post*kappa120_post/rho + 4*kappa011_post*kappa111_post/rho + 2*kappa012_post*kappa110_post/rho + kappa020_post*kappa102_post/rho + 2*kappa021_post*kappa101_post/rho)

Eq(C222_post, C222*(1 - omega[10]))

Eq(kappa222_post, C222_post - 2*kappa002_post*kappa020_post*kappa200_post/rho**2 - 4*kappa002_post*kappa110_post**2/rho**2 + kappa002_post*kappa220_post/rho - 4*kappa011_post**2*kappa200_post/rho**2 - 16*kappa011_post*kappa101_post*kappa110_post/rho**2 + 4*kappa011_post*kappa211_post/rho + 2*kappa012_post*kappa210_post/rho - 4*kappa020_post*kappa101_post**2/rho**2 + kappa020_post*kappa202_post/rho + 2*kappa021_post*kappa201_post/rho + kappa022_post*kappa200_post/rho + 4*kappa101_post*kappa121_post/rho + 2*kappa102_post*kappa120_post/rho + 4*kappa110_post*kappa112_post/rho + 4*kappa111_post**2/rho)

## Backward transformation: $\kappa_{\alpha \beta \gamma} \rightarrow m_{\alpha \beta \gamma}$

We have completed the collision operation and backward transformation from $C$ to $\kappa$. The reminaing tasks are the backward transformation from $\kappa$ to $m$ and $m$ to $f$. Here, we buid the former. The transformation is given by (see [Moment Transform](https://pycodegen.pages.i10git.cs.fau.de/lbmpy/sphinx/moment_transforms.html) in lbmpy API reference)

\begin{align}
&m^{*}_{ij|\gamma} = \sum_{k=0}^{\gamma} \left(\begin{array}{c}
        \gamma \\
        k 
    \end{array} \right) w^{\gamma-k} \kappa^{*}_{ijk} \\
&m^{*}_{i|\beta \gamma} = \sum_{j=0}^{\beta} \left(\begin{array}{c}
        \beta \\
        j 
    \end{array} \right)  v^{\beta-j} m^{*}_{ij|\gamma} \\
&m^{*}_{\alpha \beta \gamma}  = \sum_{i=0}^{\alpha} \left(\begin{array}{c}
        \alpha \\
        i 
    \end{array} \right) u^{\alpha-i} m^{*}_{i|\beta \gamma} 
\end{align}

where the binomial is defined by 

\begin{equation}
\left(
\begin{array}{c}
    \alpha \\
        i 
\end{array} 
\right)
    = \frac{\alpha!}{i! (\alpha - i)!}
\end{equation}

The chimera transformation is implemented as follows. 

<div style="
    background-color: #f0f8ff; 
    border-left: 5px solid #a2c4c9; 
    padding: 15px; 
    margin: 10px 0; 
    border-radius: 4px; 
    color: #2c4a5e;
">
    <strong>Note:</strong> The implementation here differs from that in Geier et al., in which the post-collision central moments are directly transformed into the distribution.
</div>

In [55]:
def back_trans_kappa_to_raw(dim, moment_orders, vsym, K_post, M_post, pipe_exprs=None):
    if pipe_exprs == None:
        pipe_exprs = {}
    
    current_k = {o: K_post[o] for o in moment_orders}

    if dim == 2:
        u_sym, v_sym = vsym
    else:
        u_sym, v_sym, w_sym = vsym

    if dim == 3:
        m_ij_gamma = {} # m_{ij|gamma}
        for i in [0, 1, 2]:
            for j in [0, 1, 2]:
                for gamma in [0, 1, 2]:
                    expr = 0
                    for k in range(gamma + 1):
                        binom = math.comb(gamma, k) # binomial (gamma | k)
                        u_pow = w_sym**(gamma - k) if (gamma - k) > 0 else 1
                        expr += binom * u_pow * current_k[(i, j, k)] # m_{ij|gamma} = sum (gamma | k) w^{gamma - k} kappa_{ijk}

                    var_sym = sp.Symbol(f"m_post_idx_{i}_{j}_{chimera_delimiter}_{gamma}")
                    m_ij_gamma[(i, j, gamma)] = var_sym
                    pipe_exprs[var_sym] = sp.simplify(expr)
                    
        m_i_beta_gamma = {} # m_{i|beta gamma}
        for i in [0, 1, 2]:
            for beta in [0, 1, 2]:
                for gamma in [0, 1, 2]:
                    expr = 0
                    for j in range(beta + 1):
                        binom = math.comb(beta, j) # binomial (beta | j)
                        u_pow = v_sym**(beta - j) if (beta - j) > 0 else 1
                        expr += binom * u_pow * m_ij_gamma[(i, j, gamma)] # m_{i|beta gamma} = sum (beta | j) v^{beta - j} m_{ij|gamma}

                    var_sym = sp.Symbol(f"m_post_idx_{i}_{chimera_delimiter}_{beta}_{gamma}")
                    m_i_beta_gamma[(i, beta, gamma)] = var_sym
                    pipe_exprs[var_sym] = sp.simplify(expr)
                    
        m_post_dict = {} # m_{alpha beta gamma}
        for alpha in [0, 1, 2]:
            for beta in [0, 1, 2]:
                for gamma in [0, 1, 2]:
                    expr = 0
                    for i in range(alpha + 1):
                        binom = math.comb(alpha, i) # binomial (alpha | i)
                        u_pow = u_sym**(alpha - i) if (alpha - i) > 0 else 1
                        expr += binom * u_pow * m_i_beta_gamma[(i, beta, gamma)] # m_{alpha beta gamma} ~ sum (alpha | i) u^{alpha - i} m_{i|beta gamma}

                    o = (alpha, beta, gamma)
                    pipe_exprs[M_post[o]] = sp.simplify(expr)
                    
    else: # dim == 2
        m_i_gamma = {}
        for i in [0, 1, 2]:
            for gamma in [0, 1, 2]:
                expr = 0
                for j in range(gamma + 1):
                    binom = math.comb(gamma, j)
                    u_pow = v_sym**(gamma - j) if (gamma - j) > 0 else 1
                    expr += binom * u_pow * current_k[(i, j)]

                var_sym = sp.Symbol(f"m_post_idx_{i}_{chimera_delimiter}_{gamma}")
                m_i_gamma[(i, gamma)] = var_sym
                pipe_exprs[var_sym] = sp.simplify(expr)
                
        m_post_dict = {}
        for alpha in [0, 1, 2]:
            for beta in [0, 1, 2]:
                expr = 0
                for i in range(alpha + 1):
                    binom = math.comb(alpha, i)
                    u_pow = u_sym**(alpha - i) if (alpha - i) > 0 else 1
                    expr += binom * u_pow * m_i_gamma[(i, beta)]

                o = (alpha, beta)
                pipe_exprs[M_post[o]] = sp.simplify(expr)

    return pipe_exprs

In [56]:
pipe_backward = back_trans_kappa_to_raw(dim, moment_orders, vel_syms, K_post, M_post) # backward transform: kappa to moment

Quick check for the first several chimeras. Here, `m_post_idx_0_0_c_0` means $m_{00|0}^{*}$.

In [57]:
print(f"{list(pipe_backward.items())[:5]}")

[(m_post_idx_0_0_c_0, kappa000_post), (m_post_idx_0_0_c_1, kappa000_post*w + kappa001_post), (m_post_idx_0_0_c_2, kappa000_post*w**2 + 2*kappa001_post*w + kappa002_post), (m_post_idx_0_1_c_0, kappa010_post), (m_post_idx_0_1_c_1, kappa010_post*w + kappa011_post)]


## Backward transformation: $m_{\alpha \beta \gamma} \rightarrow f_{i}$

<div style="
    background-color: #f8f9fa; 
    border-left: 5px solid #ced4da; 
    padding: 15px; 
    margin: 10px 0; 
    border-radius: 4px; 
    color: #495057;
">
    <strong>Reference:</strong> See Sec 4.3 in Geier et al. (2015): from Eq. (89)-(97).
</div>

\begin{equation}
\begin{array}{ll}
m_{0|\beta\gamma} = m_{0\beta\gamma} - m_{2\beta\gamma} &\text{(I)} \\
m_{\bar{1}|\beta\gamma} = (-m_{1\beta\gamma} + m_{2\beta\gamma}) / 2 &\text{(II)} \\
m_{1|\beta\gamma} = (m_{1\beta\gamma} + m_{2\beta\gamma}) / 2 &\text{(III)} \\
m_{i0|\gamma} = m_{i0\gamma} - m_{i2\gamma} &\text{(IV)} \\
m_{i\bar{1}|\gamma} = (-m_{i1\gamma} + m_{i2\gamma}) / 2 &\text{(V)} \\
m_{i1|\gamma} = (m_{i1\gamma} + m_{i2\gamma}) / 2 &\text{(VI)} \\
f_{ i j 0} = m_{ij|0} - m_{ij|2} &\text{(VII)} \\
f_{ i j-1} = \frac{ -m_{ij|1} + m_{ij|2} }{2} &\text{(VIII)} \\
f_{ i j 1} = \frac{  m_{ij|1} + m_{ij|2} }{2} &\text{(IX)}
\end{array}
\end{equation}

These equations are implemented in `back_trans_raw_to_f` below. See the equation labels (I)-(IX) attached to the code for comparison with the definitions.  

In [58]:
def back_trans_raw_to_f(dim, v_map, M_post, pipe_exprs=None):
    if pipe_exprs == None:
        pipe_exprs = {}

    f_post_exprs = {}

    if dim == 3:
        pm_a_b_e_g = {}
        for beta in [0, 1, 2]:
            for gamma in [0, 1, 2]:
                m0 = M_post[(0, beta, gamma)]
                m1 = M_post[(1, beta, gamma)]
                m2 = M_post[(2, beta, gamma)]
                pm0  = sp.simplify(  m0 - m2) # - * - (I) - * - #
                pmm1 = sp.simplify((-m1 + m2) / 2) # - * - (II) - * - #
                pm1  = sp.simplify(( m1 + m2) / 2) # - * - (III) - * - #
                for i, expr in [(0, pm0), (-1, pmm1), (1, pm1)]:
                    var_sym = sp.Symbol(f"{chimera_name}_m_post_{v_map[i]}_{beta}_{chimera_delimiter}_{gamma}")
                    pm_a_b_e_g[(i, beta, gamma)] = var_sym # - * - m_{i|beta gamma} - * - #
                    pipe_exprs[var_sym] = expr
                    
        pm_a_e_b_g = {}
        for i in [-1, 0, 1]:
            for gamma in [0, 1, 2]:
                pm_b0 = pm_a_b_e_g[(i, 0, gamma)]
                pm_b1 = pm_a_b_e_g[(i, 1, gamma)]
                pm_b2 = pm_a_b_e_g[(i, 2, gamma)]
                pm0  = sp.simplify(  pm_b0 - pm_b2) # - * - (IV) - * - #
                pmm1 = sp.simplify((-pm_b1 + pm_b2) / 2) # - * - (V) - * - #
                pm1  = sp.simplify(( pm_b1 + pm_b2) / 2) # - * - (VI) - * - #
                for j, expr in [(0, pm0), (-1, pmm1), (1, pm1)]:
                    var_sym = sp.Symbol(f"{chimera_name}_m_post_{v_map[i]}_{chimera_delimiter}_{v_map[j]}_{gamma}")
                    pm_a_e_b_g[(i, j, gamma)] = var_sym # - * - m_{ij|gamma} - * - #
                    pipe_exprs[var_sym] = expr
                    
        for i in [-1, 0, 1]:
            for j in [-1, 0, 1]:
                pm_g0 = pm_a_e_b_g[(i, j, 0)]
                pm_g1 = pm_a_e_b_g[(i, j, 1)]
                pm_g2 = pm_a_e_b_g[(i, j, 2)]
                f0  = sp.simplify(  pm_g0 - pm_g2) # - * - (VII) - * - #
                fm1 = sp.simplify((-pm_g1 + pm_g2) / 2) # - * - (VIII) - * - #
                f1  = sp.simplify(( pm_g1 + pm_g2) / 2) # - * - (IX) - * - # !Goal!
                for k, expr in [(0, f0), (-1, fm1), (1, f1)]:
                    f_post_sym = sp.Symbol(f"f_post_idx_{v_map[i]}_{v_map[j]}_{v_map[k]}")
                    f_post_exprs[(i, j, k)] = f_post_sym # - * - f_{ijk} - * - #
                    pipe_exprs[f_post_sym] = expr
                    
    else: # dim == 2
        pm_a_e_g = {}
        for beta in [0, 1, 2]:
            m0 = sp.Symbol(f"m0{beta}_post")
            m1 = sp.Symbol(f"m1{beta}_post")
            m2 = sp.Symbol(f"m2{beta}_post")
            pm0  = sp.simplify(  m0 - m2)
            pmm1 = sp.simplify((-m1 + m2) / 2)
            pm1  = sp.simplify(( m1 + m2) / 2)
            for i, expr in [(0, pm0), (-1, pmm1), (1, pm1)]:
                var_sym = sp.Symbol(f"{chimera_name}_m_post_{v_map[i]}_{beta}")
                pm_a_e_g[(i, beta)] = var_sym
                pipe_exprs[var_sym] = expr
                
        for i in [-1, 0, 1]:
            pm_b0 = pm_a_e_g[(i, 0)]
            pm_b1 = pm_a_e_g[(i, 1)]
            pm_b2 = pm_a_e_g[(i, 2)]
            f0  = sp.simplify(  pm_b0 - pm_b2)
            fm1 = sp.simplify((-pm_b1 + pm_b2) / 2)
            f1  = sp.simplify(( pm_b1 + pm_b2) / 2)
            for j, expr in [(0, f0), (-1, fm1), (1, f1)]:
                f_post_sym = sp.Symbol(f"f_post_idx_{v_map[i]}_{v_map[j]}")
                f_post_exprs[(i, j)] = f_post_sym
                pipe_exprs[f_post_sym] = expr

    return pipe_exprs, f_post_exprs

In [59]:
pipe_mtof, f_post_exprs = back_trans_raw_to_f(dim, v_map, M_post) # backward transform: moment to f

In [60]:
print(list(f_post_exprs.items())[:5])

[((-1, -1, 0), f_post_idx_m1_m1_0), ((-1, -1, -1), f_post_idx_m1_m1_m1), ((-1, -1, 1), f_post_idx_m1_m1_1), ((-1, 0, 0), f_post_idx_m1_0_0), ((-1, 0, -1), f_post_idx_m1_0_m1)]


In [61]:
# re-order allsignments as 
# from macro to post moment, (stored in asignments_other)
# then, post f in oder of 0, 1, 2, ... (stored in f_post)

pipe_mtof_ordered = {}

f_post_map = {}

assignments_other = []

# mapping f_{ijk} -> f_{i}
def convert_direction_to_idx(dim, var_name): 
    parts = var_name.split("_")
    def parse_v(s):
        if s == "m1": return -1
        return int(s)
    if dim == 2:
        return (parse_v(parts[3]), parse_v(parts[4])) # (i, j)
    else:
        return (parse_v(parts[3]), parse_v(parts[4]), parse_v(parts[5])) # (i, j, k)

for var_sym, expr in zip(pipe_mtof.keys(), pipe_mtof.values()):
    var_name = var_sym.name
    if var_name.startswith("f_post_idx_"):
        f_idx = vectors.index(convert_direction_to_idx(dim, var_name))
        f_post_map[f_idx] = expr # write f_new last. for this, we escape them into f_post_map
    else:
        assignments_other.append((var_name, expr))

pipe_mtof_chimera = {}

for var_name, expr in assignments_other:
    pipe_mtof_chimera[sp.Symbol(var_name)] = expr

pipe_mtof_f_post = {}

for f_idx in sorted(f_post_map.keys()):
    expr = f_post_map[f_idx]
    f_name = f"f{f_idx}_post"
    pipe_mtof_f_post[sp.Symbol(f_name)] = expr

    if f_idx < 5:
        print(f"{f_name} = {expr}")

    # print some examples of sorted f


f0_post = chimera_m_post_0_c_0_0 - chimera_m_post_0_c_0_2
f1_post = chimera_m_post_1_c_0_0 - chimera_m_post_1_c_0_2
f2_post = chimera_m_post_m1_c_0_0 - chimera_m_post_m1_c_0_2
f3_post = chimera_m_post_0_c_1_0 - chimera_m_post_0_c_1_2
f4_post = chimera_m_post_0_c_m1_0 - chimera_m_post_0_c_m1_2


Concatenete pipelines.

In [62]:
pipe_exprs = pipe_forward | pipe_collision | pipe_backward | pipe_mtof_chimera | pipe_mtof_f_post

## CSE in forward transformation

We apply Sympy CSE to the first moment chimera, and the forward transformation from $\kappa$ to $C$. 

In [63]:
chimera1st_targets = {}
for suffix, chimera_sym in zip(first_chimera.keys(), first_chimera.values()):
    chimera1st_targets[chimera_sym] = pipe_exprs_m_chimera[chimera_sym]

cse_targets_mc = ( list(chimera1st_targets.values()) )
replacements_mc, reduced_exprs_mc = sp.cse(cse_targets_mc, symbols=sp.symbols('xm0:500'))

In [64]:
for i in range(len(moment_orders)):
    o, name = moment_orders[i], mom_names[i]
    C_cum_expr[o] = sp.expand( C_cum_expr[o] )

cse_targets_cl = []
for i in range(len(moment_orders)):
    o, name = moment_orders[i], mom_names[i]
    cse_targets_cl.append( C_cum_expr[o] )

replacements_cl, reduced_exprs_cl = sp.cse(cse_targets_cl, symbols=sp.symbols('xc0:500'))

Finalize the pipeline.

In [65]:
pipe_all = {}

# [forward] raw moment
for var, expr in replacements_mc:
    pipe_all[var] = expr

for i, chimera_sym in enumerate(first_chimera.values()):
    pipe_all[chimera_sym] = reduced_exprs_mc[i]

for suffix, key in zip(second_chimera.keys(), second_chimera.values()):
    pipe_all[key] = pipe_exprs_m_chimera[key]

for i in range(len(moment_orders)):
    o, name = moment_orders[i], mom_names[i]
    pipe_all[M_raw[o]] = M_raw_expr[o]

# [macroscopic variables]
# "rho = m000", "u = m100/m000", "inv_rho = 1.0/rho" are required in the following transformation
pipe_all[rho] = M_raw[(0,0,0)[:dim]]

vel_orders = [(1,0,0)[:dim], (0,1,0)[:dim], (0,0,1)[:dim]][:dim]
for d in range(dim):
    pipe_all[vel_syms[d]] = M_raw[vel_orders[d]]/rho

# [forward] central moment # non CSE for kappa reconstruction
kappa_subs_map = {}
for o in moment_orders:
    if sum(o) > 1:
        order = ''.join(str(x) for x in o)
        k_name = "\kappa_{" + order + "}"
        kappa_subs_map.update({k_name: K_cen[o]})

for o in moment_orders:
    if sum(o) > 1:
        pipe_all[K_cen[o]] = K_cen_expr[o].subs(kappa_subs_map)

# [forward] cumulant
for var, expr in replacements_cl:
    pipe_all[var] = expr

for i in range(len(moment_orders)):
    o, name = moment_orders[i], mom_names[i]
    if sum(o) > 3:
        pipe_all[C_cum[o]] = reduced_exprs_cl[i]

# [collision]
for key, expr in zip(pipe_collision.keys(), pipe_collision.values()):
    pipe_all[key] = expr

# [backward] kappa to m
for key, expr in zip(pipe_backward.keys(), pipe_backward.values()):
    k_post_subs_map = {K_post[(0,0,0)[:dim]]: rho, K_post[(1,0,0)[:dim]]: 0, K_post[(0,1,0)[:dim]]: 0}
    if dim == 3:
        k_post_subs_map.update({K_post[(0,0,1)[:dim]]: 0})
    expr = expr.subs(k_post_subs_map)
    pipe_all[key] = expr

# [backward] m to f
for key, expr in zip(pipe_mtof_chimera.keys(), pipe_mtof_chimera.values()):
    pipe_all[key] = expr

In [66]:
#pipe_all | pipe_mtof_f_post # remove comment '#' to check the result

## Export pipeline as Taichi kernel

In [67]:
idt = ['', '    ', '        ']

print(f"{idt[0]}@ti.kernel")
print(f"{idt[0]}def col_stream_core(rho: ti.template(), vel: ti.template(), f_pre: ti.template(), f_post: ti.template()):")

print(f"{idt[1]}for I in ti.grouped(rho):")

for i in range(num_pops):
    print(f"{idt[2]}{f_syms[i]} = f_pre[I-c[{i}]][{i}]")

for var, expr in pipe_all.items():
    print(f"{idt[2]}{var} = {expr}")

for i, key in enumerate(pipe_mtof_f_post):
    
    print(f"{idt[2]}f_post[I][{i}] = {pipe_mtof_f_post[key]}")

print(f"{idt[2]}{rho}[I] = {rho}")

for d_idx in range(dim):
    print(f"{idt[2]}vel[I][{d_idx}] = {vel_syms[d_idx]}")

@ti.kernel
def col_stream_core(rho: ti.template(), vel: ti.template(), f_pre: ti.template(), f_post: ti.template()):
    for I in ti.grouped(rho):
        f0 = f_pre[I-c[0]][0]
        f1 = f_pre[I-c[1]][1]
        f2 = f_pre[I-c[2]][2]
        f3 = f_pre[I-c[3]][3]
        f4 = f_pre[I-c[4]][4]
        f5 = f_pre[I-c[5]][5]
        f6 = f_pre[I-c[6]][6]
        f7 = f_pre[I-c[7]][7]
        f8 = f_pre[I-c[8]][8]
        f9 = f_pre[I-c[9]][9]
        f10 = f_pre[I-c[10]][10]
        f11 = f_pre[I-c[11]][11]
        f12 = f_pre[I-c[12]][12]
        f13 = f_pre[I-c[13]][13]
        f14 = f_pre[I-c[14]][14]
        f15 = f_pre[I-c[15]][15]
        f16 = f_pre[I-c[16]][16]
        f17 = f_pre[I-c[17]][17]
        f18 = f_pre[I-c[18]][18]
        f19 = f_pre[I-c[19]][19]
        f20 = f_pre[I-c[20]][20]
        f21 = f_pre[I-c[21]][21]
        f22 = f_pre[I-c[22]][22]
        f23 = f_pre[I-c[23]][23]
        f24 = f_pre[I-c[24]][24]
        f25 = f_pre[I-c[25]][25]
        f26 = f_pre[I-c[2

Great!

You can derive D2Q9 kernel just by setting `dim = 2`.

Additional things not discussed here, but considered in `lb_solver/`:

- replace `1/rho` with `inv_rho`
- replace rational numbers, e.g., replace `1/2` with `* 0.5`, `1/6` with `INV_6`, and so on.
- pull/push streaming
- $\delta \rho$ mode
- CSE $f^{eq}$ (may be used in initial and boundary conditions)